# 03 · Feature families

Five ways to turn epochs into something a classifier can use:

- **Time:** Hjorth params, moments, entropy
- **Frequency:** log band power (δ θ α β γ)
- **Connectivity:** phase-locking between channels
- **CSP / FBCSP:** spatial filters tuned to the discriminative bands
- **Riemannian:** trial covariance matrices on the SPD manifold

Each is an sklearn transformer, so it stays *inside* the pipeline → no leakage.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2
import numpy as np, matplotlib.pyplot as plt


In [ ]:
from eeglog.data import load_moabb
from eeglog import features as F
d = load_moabb(subjects=[1], paradigm='left_right')
X, y, sf = d.X, d.y, d.sfreq

In [ ]:
# Time + frequency feature matrices.
Xt = F.TimeDomainFeatures(sf).fit_transform(X)
Xf = F.BandPowerFeatures(sf).fit_transform(X)
print('time:', Xt.shape, '| bandpower:', Xf.shape)

In [ ]:
# CSP spatial patterns — the classic MI artifact. Topomaps differ by class.
import mne
csp = F.make_csp(n_components=4).fit(X, y)
info = mne.create_info([f'C{i}' for i in range(X.shape[1])], sf, 'eeg')
# (For real topomaps use the dataset montage; here we just show CSP works.)
print('CSP feature shape:', csp.transform(X).shape)

In [ ]:
# Band power by class — alpha/beta desync should differ left vs right.
import numpy as np
for cls in sorted(set(y)):
    plt.plot(Xf[y == cls].mean(0), label=cls, lw=1)
plt.legend(); plt.title('mean log band power by class'); plt.xlabel('feature idx')

**Checkpoint:** five feature families, each an sklearn transformer. Next we plug them into classical models.